# Testing Improved Linear Regression Models
## Direct Prediction vs. Residual Learning

This notebook tests the improved linear regression models that fix the core flaw in the original implementation.

### Key Difference
- **Original**: Predicts residuals FROM context mean → Can't beat baseline
- **Improved**: Predicts ABSOLUTE cost_percent → Can be competitive with baseline

### Expected Results
- DirectPredictionLinearRegression: Should be ≥ baseline performance
- DirectPredictionRidgeRegression: Similar or better with regularization
- HybridBaselineLinearRegression: Safest approach, blends both

### Setup
This notebook reuses the data and evaluation infrastructure from the main notebook,
but tests the improved models.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge

print("Dependencies imported successfully")

## Step 1: Load and Prepare Data

First, load the data from the main notebook (or recompute if needed).

In [ ]:
# ============================================================================
# LOAD DATA (copy relevant setup from main notebook)
# ============================================================================

# Configuration
ID_COLS = ['split_year', 'week_of_year', 'merchant_id']
CONTEXT_WEEKS = [1, 2, 3, 4]  # First 4 weeks
EVAL_WEEKS_SCENARIO = list(range(5, 17))  # Weeks 5-16 (12 weeks to predict)
MCC_TARGET = 5411
DATASET_REPRESENTATION = 'fraction'
REPRESENTATION_PARAM = 0.1  # Use 10% of merchants for faster testing

# NOTE: This cell assumes you have the processed_transactions_4mcc.csv file
# If running in isolation, you'll need to either:
# 1. Run the main notebook first to generate data
# 2. Load from saved pickle/parquet

# Placeholder: In real execution, load the actual data
print("Data loading would happen here...")
print(f"MCC Target: {MCC_TARGET}")
print(f"Context weeks: {CONTEXT_WEEKS}")
print(f"Eval weeks: {EVAL_WEEKS_SCENARIO}")

## Step 2: Define Improved Model Classes

In [ ]:
class DirectPredictionLinearRegression:
    """
    Week-wise linear regression predicting ABSOLUTE cost_percent values.
    
    KEY IMPROVEMENT: Predicts cost_percent directly, not residuals
    - Traditional: y = cost_percent - context_mean (residual)
    - Improved: y = cost_percent (absolute value)
    """
    
    def __init__(self, feature_cols, id_cols=None):
        if id_cols is None:
            id_cols = ID_COLS
        self.feature_cols = feature_cols
        self.id_cols = id_cols
        self.models = {}  # week -> LinearRegression model
        self.pool_means_by_week = {}  # fallback estimates
        self.is_trained = False
    
    def train(self, pool_weekly_df):
        """Train models predicting absolute cost_percent."""
        self.models = {}
        self.pool_means_by_week = {}
        
        for source_week in range(1, 53):
            target_week = source_week + 1 if source_week < 52 else 1
            
            source_features = [f for f in self.feature_cols if f != 'cost_percent']
            source_data = pool_weekly_df[
                pool_weekly_df['week_of_year'] == source_week
            ][['merchant_id'] + source_features].copy()
            
            target_data = pool_weekly_df[
                pool_weekly_df['week_of_year'] == target_week
            ][['merchant_id', 'cost_percent']].copy()
            
            if source_data.empty or target_data.empty:
                self.pool_means_by_week[target_week] = 0.5
                continue
            
            merged = source_data.merge(target_data, on='merchant_id', how='inner')
            
            if len(merged) < 2:
                self.pool_means_by_week[target_week] = 0.5
                continue
            
            self.pool_means_by_week[target_week] = merged['cost_percent'].mean()
            
            X = merged[source_features].fillna(0).values
            y = merged['cost_percent'].fillna(0).values  # ← ABSOLUTE, not residual
            
            model = LinearRegression()
            model.fit(X, y)
            self.models[source_week] = model
        
        self.is_trained = True
        print(f'✓ Trained {len(self.models)} direct prediction LR models')
    
    def predict(self, query_merchant_id, query_data, context_weeks, eval_weeks, 
                all_weekly_df, feature_cols):
        """Predict using direct absolute value predictions."""
        if not self.is_trained:
            raise ValueError('Model must be trained before prediction')
        
        pred_features = [f for f in self.feature_cols if f != 'cost_percent']
        predictions = []
        
        for eval_week in eval_weeks:
            source_week = eval_week - 1 if eval_week > 1 else 52
            
            if source_week not in self.models:
                mean_for_week = self.pool_means_by_week.get(eval_week, 0.5)
                predictions.append(mean_for_week)
                continue
            
            q = query_data.loc[query_data['week_of_year'] == source_week, pred_features]
            
            if not q.empty:
                features = q.iloc[0].fillna(0).values
            else:
                features = np.zeros(len(pred_features))
            
            try:
                pred = float(self.models[source_week].predict(features.reshape(1, -1))[0])
                pred = np.clip(pred, 0, 1)  # Valid range
            except:
                pred = self.pool_means_by_week.get(eval_week, 0.5)
            
            predictions.append(pred)
        
        return np.array(predictions)

print("DirectPredictionLinearRegression defined")

In [ ]:
class DirectPredictionRidgeRegression:
    """
    Week-wise Ridge regression predicting ABSOLUTE cost_percent values.
    
    Like DirectPredictionLinearRegression but with L2 regularization.
    """
    
    def __init__(self, feature_cols, alpha=1.0, id_cols=None):
        if id_cols is None:
            id_cols = ID_COLS
        self.feature_cols = feature_cols
        self.alpha = alpha
        self.id_cols = id_cols
        self.models = {}
        self.pool_means_by_week = {}
        self.is_trained = False
    
    def train(self, pool_weekly_df):
        """Train Ridge models predicting absolute cost_percent."""
        self.models = {}
        self.pool_means_by_week = {}
        
        for source_week in range(1, 53):
            target_week = source_week + 1 if source_week < 52 else 1
            
            source_features = [f for f in self.feature_cols if f != 'cost_percent']
            source_data = pool_weekly_df[
                pool_weekly_df['week_of_year'] == source_week
            ][['merchant_id'] + source_features].copy()
            
            target_data = pool_weekly_df[
                pool_weekly_df['week_of_year'] == target_week
            ][['merchant_id', 'cost_percent']].copy()
            
            if source_data.empty or target_data.empty:
                self.pool_means_by_week[target_week] = 0.5
                continue
            
            merged = source_data.merge(target_data, on='merchant_id', how='inner')
            
            if len(merged) < 2:
    

## Comparison: Original vs. Improved Strategy

### Original Residual Learning (FLAWED)
```
Training:
  For each merchant in pool:
    context_mean = mean(cost_percent for weeks 1-4)
    For week 5: y = cost_percent_week5 - context_mean
    For week 6: y = cost_percent_week6 - context_mean
    ...
  Model learns: features_weekN → residual_from_mean

Prediction:
  prediction = merchant_context_mean + predicted_residual
  
  Problem: By design, residuals are centered around 0
  - Any deviation → worse than baseline
  - Model competes with the residual distribution, not cost_percent
  - Baseline already uses context_mean!
```

### Improved Direct Prediction (CORRECT)
```
Training:
  For each merchant in pool:
    For week 5: y = cost_percent_week5 (absolute value)
    For week 6: y = cost_percent_week6 (absolute value)
    ...
  Model learns: features_weekN → cost_percent_weekN+1

Prediction:
  prediction = learned_model(features_weekN)
  
  Benefit: Model competes with actual cost_percent values
  - Can capture trends better than fixed baseline
  - Learns pool-wide weekly patterns
  - More stable coefficient estimates
```

## Integration with Main Notebook

To use these improved models with the existing evaluation framework:

```python
# 1. Save improved models to a module
# File: improved_lr_models.py (already created)

# 2. Register improved models in main notebook
from improved_lr_models import (
    DirectPredictionLinearRegression,
    DirectPredictionRidgeRegression,
    HybridBaselineLinearRegression
)

# Add to MODEL_REGISTRY
MODEL_REGISTRY['direct_linear_regression'] = {
    'model': DirectPredictionLinearRegression(feature_cols),
    'description': 'Direct prediction of cost_percent (absolute value)'
}

MODEL_REGISTRY['direct_ridge_regression'] = {
    'model': DirectPredictionRidgeRegression(feature_cols, alpha=1.0),
    'description': 'Ridge regression predicting absolute cost_percent'
}

MODEL_REGISTRY['hybrid_baseline_lr'] = {
    'model': HybridBaselineLinearRegression(feature_cols),
    'description': 'Hybrid: 60% baseline mean + 40% learned prediction'
}

# 3. Run evaluation with new models
# All existing evaluation code will work with the new models
# Because they implement the same interface: predict(...) method
```

### Expected Results
```
Current (Residual-based):
  baseline_constant_mean:        MAE = 0.2748 ✓ BEST
  weekwise_linear_regression:    MAE = 0.3688 ✗ WORST
  weekwise_ridge_regression:     MAE = 0.3688 ✗ WORST

Expected with Improved (Direct Prediction):
  baseline_constant_mean:        MAE = 0.2748 (unchanged)
  direct_linear_regression:      MAE ≈ 0.27-0.30 (should improve!)
  direct_ridge_regression:       MAE ≈ 0.26-0.29 (possibly best)
  hybrid_baseline_lr:            MAE ≈ 0.27-0.29 (safe compromise)
  
Key metric: We need Direct LR MAE < 0.30 to show improvement
```

## Key Files & References

### Root Cause Analysis
- **File**: [LINEAR_REGRESSION_ROOT_CAUSE_ANALYSIS.md](LINEAR_REGRESSION_ROOT_CAUSE_ANALYSIS.md)
- **Summary**: Detailed explanation of 6 root causes, why baseline wins, and solution recommendations

### Improved Model Implementation
- **File**: [improved_lr_models.py](improved_lr_models.py)
- **Classes**:
  - `DirectPredictionLinearRegression`: Basic improved model
  - `DirectPredictionRidgeRegression`: With regularization
  - `HybridBaselineLinearRegression`: Blend baseline + learned (safest)

### Original Notebook with Issue
- **File**: [Unified SARIMA vs Supervised Benchmark.ipynb](Unified%20SARIMA%20vs%20Supervised%20Benchmark.ipynb)
- **Issue Location**: Lines 1008-1324 (WeekwiseLinearRegression, WeekwiseRidgeRegression)
- **Evaluation**: Lines 1384-1534
- **Results**: Cells 22-24

### Testing This Notebook
To run a full test with actual data:
1. Load `working_weekly_df` and `feature_cols` from the main notebook
2. Uncomment the data loading sections above
3. Register the improved models
4. Run the evaluation
5. Compare results